# 55 — OpenRouter Setup
**Goal:** Configure OpenRouter API, select models, manage costs and rate limits.

The previous chapters built the rule-based and semantic layers of the ATS entirely in-process. This chapter opens the door to the **LLM layer**: a single OpenAI-compatible client that fronts 200+ hosted models — GPT, Claude, Gemini, Llama, Mistral — through one endpoint. Everything downstream in this block (prompt templates, structured JSON, tool calling) speaks to that one client, so getting the plumbing right here pays off for the next eight chapters.

**Why it matters for resumes / ATS:** resume rewriting, skill classification, and STAR generation are *generation* tasks where fixed rules plateau — no grammar can produce fluent, quantified bullets. A hosted LLM gateway lets the ATS pick the cheapest adequate model per task, fall back when one provider throttles, and meter every call against a single account. One API, many models, no vendor lock-in — that is the infrastructure every later chapter assumes.

## 1. Why OpenRouter?

OpenRouter is an **aggregator, not a model**: it proxies requests to many providers behind one OpenAI-compatible endpoint. That buys three properties a production ATS needs — **portability** (swap `gpt-4o-mini` for `claude-3.5-haiku` by changing one string), **resilience** (automatic fallback when a provider is down or rate-limited), and **cost control** (per-model pricing and usage tracking on one dashboard).

**What the code does:** prints the pitch — one endpoint for 200+ models, built-in fallback, cost tracking, no vendor lock-in — then maps each resume task family to a concrete model: `GPT-4-mini` for rewriting and generation, `Claude 3 Haiku` for cheap classification, `Llama 3 70B` as the open-source alternative, `Mistral Small` as the budget option. That mapping is the seed of the `pick_model()` helper in section 3.

In [ ]:
print('''OpenRouter provides unified API to 200+ LLMs:
- Single endpoint for GPT-4, Claude, Gemini, Llama, Mistral, etc.
- Built-in fallback if a model is rate-limited or down
- Cost tracking per model
- No vendor lock-in

For resume analysis:
  - GPT-4-mini: Best for rewriting, generation
  - Claude 3 Haiku: Fast, cheap for classification
  - Llama 3 70B: Open-source alternative
  - Mistral Small: Budget option for simple tasks''')

## 2. Setting Up the Client

The client is the standard OpenAI SDK pointed at a different `base_url`. The only real configuration is the **API key**, which should come from the environment — never hard-coded into a notebook that might be committed.

**What the code does:**
- `os.getenv("OPENROUTER_API_KEY")` — reads the key from the environment, falling back to `OPENAI_API_KEY`; the same client object works for either because the base URL decides the provider.
- `OpenAI(base_url="https://openrouter.ai/api/v1", api_key=...)` — an OpenAI-compatible client that speaks the Chat Completions protocol to OpenRouter.
- `client.models.list()` — the connection test. **Expected:** with a valid key it returns the model catalog and the cell prints the first five free model ids (`... if "free" in m.id`); without a key it raises an auth error and the `except` branch prints the message plus the "(Expected if no API key...)" note, so the notebook still runs as reference material.

In [ ]:
# pip install openai
from openai import OpenAI
import os

# Get API key from environment
api_key = os.getenv("OPENROUTER_API_KEY") or os.getenv("OPENAI_API_KEY")
if not api_key:
    api_key = ""  # Fill in your key here
    print("⚠ No API key found. Set OPENROUTER_API_KEY in .env or enter directly.")
    print("  Get one at: https://openrouter.ai/keys")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

# Test the connection
try:
    response = client.models.list()
    model_ids = [m.id for m in response.data if "free" in m.id][:5]
    print(f"✓ API connected. Free models available: {model_ids}")
except Exception as e:
    print(f"Connection test: {e}")
    print("(Expected if no API key — notebook works as reference.)")

## 3. Model Selection Helper

Model choice is a **cost/quality trade**, so the code codifies it as three named tiers — `cheap`, `balanced`, `best`, each with two interchangeable model ids — then maps *task families* to tiers.

**What the code does:**
- `MODEL_TIERS` — dict of tier — model list; the first entry of a tier is the default.
- `pick_model(task, quality="balanced")` — looks up the task's recommended tier (`classification — cheap`, `extraction — balanced`, `rewriting — balanced`, `generation — best`) and returns that tier's first model; unknown tasks fall back to the caller's `quality` argument.

**Expected (verified by running):** the loop prints `classification -> mistralai/mistral-small-24b-instruct-2501`, `extraction -> openai/gpt-4o-mini`, `rewriting -> openai/gpt-4o-mini`, `generation -> openai/gpt-4o`. The pattern to internalize: classification is the only task deemed cheap enough for a small model — generation, where quality shows, always gets the flagship.

In [ ]:
MODEL_TIERS = {
    "cheap": ["mistralai/mistral-small-24b-instruct-2501", "google/gemini-2.0-flash-lite-001"],
    "balanced": ["openai/gpt-4o-mini", "anthropic/claude-3.5-haiku"],
    "best": ["openai/gpt-4o", "anthropic/claude-3.5-sonnet"],
}

def pick_model(task, quality="balanced"):
    """Pick the right model for the task."""
    task_recommendations = {
        "classification": "cheap",
        "extraction": "balanced",
        "rewriting": "balanced",
        "generation": "best",
    }
    tier = task_recommendations.get(task, quality)
    return MODEL_TIERS[tier][0]

for task in ["classification", "extraction", "rewriting", "generation"]:
    print(f"  {task:15s} -> {pick_model(task)}")

## 4. Cost Tracking

LLM pricing is **per-token and asymmetric**: output tokens typically cost 4—10x input tokens. Before shipping a pipeline you need a rough per-call budget, which is what this cell computes from a hard-coded price table.

**What the code does:** `COST_PER_1K` holds USD per 1K tokens for three models; `estimate_cost(model, in, out)` applies `(in/1000 * price_in) + (out/1000 * price_out)` and returns a formatted `$...` string — or `"Check pricing page"` for models not in the table.

**Expected (verified by running):** for a typical resume rewrite (500 input / 300 output tokens) the estimates are `gpt-4o-mini $0.000255`, `claude-3.5-haiku $0.000500`, `mistral-small-24b $0.000280`. Two lessons: per-call costs are sub-cent (which is why LLM ATS features are viable), but at scale — 10,000 resumes times several calls each — they add up, so the tier system of section 3 exists to keep the expensive model on the expensive tasks.

In [ ]:
COST_PER_1K = {
    "openai/gpt-4o-mini": {"input": 0.000150, "output": 0.000600},
    "anthropic/claude-3.5-haiku": {"input": 0.000250, "output": 0.001250},
    "mistralai/mistral-small-24b": {"input": 0.000200, "output": 0.000600},
}

def estimate_cost(model, input_tokens, output_tokens):
    costs = COST_PER_1K.get(model)
    if not costs:
        return "Check pricing page"
    cost = (input_tokens / 1000 * costs["input"]) + (output_tokens / 1000 * costs["output"])
    return f"${cost:.6f}"

# Estimate for a typical resume
print("Cost estimate (resume rewrite, ~500 in / ~300 out):")
for model in COST_PER_1K:
    print(f"  {model:35s}: {estimate_cost(model, 500, 300)}")

## Summary: OpenRouter gives access to all major models through one API. Pick model by task complexity.

**One OpenAI-compatible client is the entire LLM infrastructure — model choice is a routing decision, not an integration decision.**

With a single `base_url`, the ATS can switch providers by editing a string, fall back when a vendor throttles, and meter cost per call. `pick_model()` turns that freedom into a policy: cheap models for classification, balanced for extraction and rewriting, flagship for generation. The numbers back it up — a single rewrite costs about a quarter of a millicent — so the bottleneck is never the API bill; it is prompt quality.

This chapter feeds Ch. 56, where the same client starts consuming carefully engineered prompts for resume-domain tasks.